In [ ]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX


HAS_XGB = True
try:
    from xgboost import XGBRegressor
except Exception:
    HAS_XGB = False


DATA_PATH = '/content/nitrous-oxide-emissions.csv'
JSON_PATH = '/content/nitrous_emissions_predictions_v4_auto_model.json'

FUTURE_YEARS = 5
TEST_SIZE = 5
N_SPLITS = 3
ACC_THRESHOLD = 80.0
EPS = 1e-8

MIN_ROWS_PER_COUNTRY = 25
LAST_N_YEARS = 50


LAGS = [1, 2, 3, 5, 8, 12]
ROLL_WINDOWS = [3, 5, 8]


SARIMAX_ORDER = (1, 1, 1)
SARIMAX_SEASONAL = (0, 0, 0, 0)

USE_LOG_TARGET = True


df = pd.read_csv("/content/nitrous-oxide-emissions.csv")


possible_targets = [
    "Nitrous oxide",
    "Nitrous oxide emissions",
    "Nitrous oxide emissions from waste",
    "nitrous_oxide",
    "nitrous_oxides",
    "N2O emissions",
    "N2O"
]
target_col = None
for c in possible_targets:
    if c in df.columns:
        target_col = c
        break

if target_col is None:
    target_col = df.select_dtypes(include='number').columns[-1]


needed_cols = ['Entity', 'Year', target_col]
if 'Code' in df.columns:
    needed_cols.append('Code')

df = df[needed_cols].copy()
df = df.dropna(subset=['Entity', 'Year', target_col])

df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df[target_col] = pd.to_numeric(df[target_col], errors='coerce')
df = df.dropna(subset=['Year', target_col])
df = df[df[target_col] >= 0].copy()

aggregate_entities = {
    "World", "Africa", "Asia", "Europe", "North America", "South America", "Oceania",
    "High-income countries", "Upper-middle-income countries",
    "Lower-middle-income countries", "Low-income countries", "Other",
    "European Union (27)", "European Union (28)",
    "Europe (excl. EU-27)", "Europe (excl. EU-28)",
    "North America (excl. USA)", "Asia (excl. China and India)"
}
df = df[~df['Entity'].isin(aggregate_entities)].copy()


group_cols = ['Entity', 'Year'] + (['Code'] if 'Code' in df.columns else [])
df = df.groupby(group_cols, as_index=False)[target_col].mean()


def winsorize_group(g, col, ql=0.01, qh=0.99):
    lo, hi = g[col].quantile(ql), g[col].quantile(qh)
    g[col] = g[col].clip(lo, hi)
    return g

df = df.groupby('Entity', group_keys=False).apply(winsorize_group, col=target_col)


counts = df.groupby('Entity')['Year'].count()
valid_entities = counts[counts >= MIN_ROWS_PER_COUNTRY].index
df = df[df['Entity'].isin(valid_entities)].copy()

df = df.sort_values(['Entity', 'Year']).reset_index(drop=True)

print("Target:", target_col)
print("Shape after cleaning:", df.shape)
print("Countries:", df['Entity'].nunique())
print("XGBoost available:", HAS_XGB)


def wmape_accuracy(y_true, y_pred):
    den = np.sum(np.abs(y_true))
    if den < EPS:
        return None
    wmape = np.sum(np.abs(y_true - y_pred)) / (den + EPS)
    return max(0.0, (1.0 - wmape) * 100.0)

def build_features(series_model_scale, years):
    s = pd.Series(series_model_scale)
    feat = pd.DataFrame(index=s.index)

    for lag in LAGS:
        feat[f'lag_{lag}'] = s.shift(lag)

    for w in ROLL_WINDOWS:
        feat[f'roll_mean_{w}'] = s.shift(1).rolling(w).mean()
        feat[f'roll_std_{w}'] = s.shift(1).rolling(w).std()

    feat['yoy_diff'] = s.diff(1)
    feat['yoy_pct'] = (s.diff(1) / (s.shift(1) + EPS)).replace([np.inf, -np.inf], np.nan)

    y = years.flatten()
    feat['year'] = y
    feat['year_idx'] = np.arange(len(y))

    feat = feat.dropna().copy()
    idx = feat.index.values
    return feat, idx

def get_ml_model(name):
    if name == 'ridge':
        return Ridge(alpha=1.2, random_state=42)
    if name == 'rf':
        return RandomForestRegressor(
            n_estimators=400,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        )
    if name == 'xgb' and HAS_XGB:
        return XGBRegressor(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.04,
            subsample=0.9,
            colsample_bytree=0.9,
            objective='reg:squarederror',
            random_state=42
        )
    return None

def one_step_features(history_vals_model, next_year):
    s = pd.Series(history_vals_model)
    row = {}

    for lag in LAGS:
        row[f'lag_{lag}'] = s.iloc[-lag] if len(s) >= lag else np.nan

    for w in ROLL_WINDOWS:
        if len(s) >= w:
            row[f'roll_mean_{w}'] = s.iloc[-w:].mean()
            row[f'roll_std_{w}'] = s.iloc[-w:].std()
        else:
            row[f'roll_mean_{w}'] = np.nan
            row[f'roll_std_{w}'] = np.nan

    if len(s) >= 2:
        row['yoy_diff'] = s.iloc[-1] - s.iloc[-2]
        row['yoy_pct'] = (s.iloc[-1] - s.iloc[-2]) / (s.iloc[-2] + EPS)
    else:
        row['yoy_diff'] = np.nan
        row['yoy_pct'] = np.nan

    row['year'] = next_year
    row['year_idx'] = len(history_vals_model)

    x = pd.DataFrame([row]).fillna(0.0)
    return x


def evaluate_sarimax_cv(y_raw):
    tscv = TimeSeriesSplit(n_splits=N_SPLITS)
    maes, rmses, accs = [], [], []

    for tr, te in tscv.split(y_raw):
        if len(te) < TEST_SIZE:
            continue
        te = te[:TEST_SIZE]

        y_train = y_raw[tr]
        y_test = y_raw[te]

        try:
            model = SARIMAX(
                y_train,
                order=SARIMAX_ORDER,
                seasonal_order=SARIMAX_SEASONAL,
                enforce_stationarity=False,
                enforce_invertibility=False
            )
            res = model.fit(disp=False)
            y_pred = np.array(res.forecast(steps=TEST_SIZE), dtype=float)
            y_pred = np.clip(y_pred, 0, None)

            mae = mean_absolute_error(y_test, y_pred)
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            acc = wmape_accuracy(y_test, y_pred)
            if acc is None:
                continue

            maes.append(mae); rmses.append(rmse); accs.append(acc)
        except Exception:
            continue

    return maes, rmses, accs

def evaluate_ml_cv(y_raw, years, model_name):
    y_model = np.log1p(y_raw) if USE_LOG_TARGET else y_raw.copy()

    X_all, idx = build_features(y_model, years.reshape(-1, 1))
    y_sup = y_model[idx]

    if len(X_all) < TEST_SIZE + 10:
        return [], [], []

    tscv = TimeSeriesSplit(n_splits=N_SPLITS)
    maes, rmses, accs = [], [], []

    for tr, te in tscv.split(X_all):
        if len(te) < TEST_SIZE:
            continue
        te = te[:TEST_SIZE]

        X_tr = X_all.iloc[tr]
        y_tr = y_sup[tr]
        X_te = X_all.iloc[te]
        y_te_model = y_sup[te]

        model = get_ml_model(model_name)
        if model is None:
            continue

        try:
            model.fit(X_tr, y_tr)
            y_pred_model = model.predict(X_te)

            if USE_LOG_TARGET:
                y_te = np.expm1(y_te_model)
                y_pred = np.expm1(y_pred_model)
            else:
                y_te = y_te_model
                y_pred = y_pred_model

            y_pred = np.clip(y_pred, 0, None)

            mae = mean_absolute_error(y_te, y_pred)
            rmse = np.sqrt(mean_squared_error(y_te, y_pred))
            acc = wmape_accuracy(y_te, y_pred)
            if acc is None:
                continue

            maes.append(mae); rmses.append(rmse); accs.append(acc)
        except Exception:
            continue

    return maes, rmses, accs


def forecast_sarimax_full(y_raw, last_year, horizon):
    model = SARIMAX(
        y_raw,
        order=SARIMAX_ORDER,
        seasonal_order=SARIMAX_SEASONAL,
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    res = model.fit(disp=False)
    pred = np.array(res.forecast(steps=horizon), dtype=float)
    pred = np.clip(pred, 0, None)
    future_years = np.arange(last_year + 1, last_year + 1 + horizon)
    return future_years, pred

def forecast_ml_full(y_raw, years, last_year, horizon, model_name):
    y_model = np.log1p(y_raw) if USE_LOG_TARGET else y_raw.copy()
    X_all, idx = build_features(y_model, years.reshape(-1, 1))
    y_sup = y_model[idx]

    model = get_ml_model(model_name)
    model.fit(X_all, y_sup)

    future_years = np.arange(last_year + 1, last_year + 1 + horizon)
    hist_vals = list(y_model.copy())
    preds_model = []

    for fy in future_years:
        x = one_step_features(hist_vals, fy)
        p = float(model.predict(x)[0])
        preds_model.append(p)
        hist_vals.append(p)

    preds_model = np.array(preds_model)

    if USE_LOG_TARGET:
        preds = np.expm1(preds_model)
    else:
        preds = preds_model

    preds = np.clip(preds, 0, None)
    return future_years, preds


dropped_countries = []
predictions_list = []
drop_reasons = {
    "accuracy_below_80": 0,
    "too_few_rows": 0,
    "too_few_samples": 0,
    "insufficient_valid_folds": 0,
    "all_models_failed": 0
}

candidate_models = ['sarimax', 'ridge', 'rf'] + (['xgb'] if HAS_XGB else [])

for country in df['Entity'].unique():
    data = df[df['Entity'] == country].sort_values('Year').copy()


    if len(data) > LAST_N_YEARS:
        data = data.tail(LAST_N_YEARS).copy()

    if len(data) < MIN_ROWS_PER_COUNTRY:
        drop_reasons["too_few_rows"] += 1
        continue

    y_raw = data[target_col].values.astype(float)
    years = data['Year'].values.astype(int)

    if len(y_raw) < (TEST_SIZE + 12):
        drop_reasons["too_few_samples"] += 1
        continue

    model_scores = []

    for mname in candidate_models:
        if mname == 'sarimax':
            maes, rmses, accs = evaluate_sarimax_cv(y_raw)
        else:
            maes, rmses, accs = evaluate_ml_cv(y_raw, years, mname)

        if len(accs) < 2:
            continue

        model_scores.append({
            "model": mname,
            "cv_folds_used": int(len(accs)),
            "mae_cv_avg": float(np.mean(maes)),
            "rmse_cv_avg": float(np.mean(rmses)),
            "accuracy_cv_avg": float(np.mean(accs)),
            "accuracy_cv_median": float(np.median(accs)),
            "accuracy_cv_std": float(np.std(accs))
        })

    if len(model_scores) == 0:
        drop_reasons["all_models_failed"] += 1
        dropped_countries.append({
            "country": country,
            "accuracy": None,
            "reason": "all_models_failed"
        })
        continue


    model_scores = sorted(
        model_scores,
        key=lambda d: (d["accuracy_cv_median"], d["accuracy_cv_avg"]),
        reverse=True
    )
    best = model_scores[0]


    if best["accuracy_cv_median"] < ACC_THRESHOLD:
        drop_reasons["accuracy_below_80"] += 1
        dropped_countries.append({
            "country": country,
            "accuracy": float(best["accuracy_cv_median"]),
            "reason": "accuracy_below_80",
            "best_model_tried": best["model"]
        })
        continue


    try:
        if best["model"] == "sarimax":
            future_years, future_pred = forecast_sarimax_full(
                y_raw=y_raw,
                last_year=int(years.max()),
                horizon=FUTURE_YEARS
            )
        else:
            future_years, future_pred = forecast_ml_full(
                y_raw=y_raw,
                years=years,
                last_year=int(years.max()),
                horizon=FUTURE_YEARS,
                model_name=best["model"]
            )
    except Exception:
        drop_reasons["all_models_failed"] += 1
        dropped_countries.append({
            "country": country,
            "accuracy": float(best["accuracy_cv_median"]),
            "reason": "final_fit_failed",
            "best_model_tried": best["model"]
        })
        continue

    predictions_list.append({
        "country": country,
        "years_used": int(len(data)),
        "selected_model": best["model"],
        "cv_folds_used": best["cv_folds_used"],
        "mae_cv_avg": best["mae_cv_avg"],
        "rmse_cv_avg": best["rmse_cv_avg"],
        "accuracy_cv_avg": best["accuracy_cv_avg"],
        "accuracy_cv_median": best["accuracy_cv_median"],
        "accuracy_cv_std": best["accuracy_cv_std"],
        "all_model_scores": model_scores,
        "future_predictions": [
            {"year": int(y), "predicted_nitrous_emissions": float(p)}
            for y, p in zip(future_years, future_pred)
        ]
    })

    print("=" * 60)
    print("Country:", country)
    print("Selected model:", best["model"])
    print(f"Years used: {len(data)}")
    print(f"CV folds used: {best['cv_folds_used']}")
    print(f"Accuracy (CV median): {best['accuracy_cv_median']:.2f}%")
    print(pd.DataFrame({
        "Year": future_years,
        "Predicted_nitrous_emissions": future_pred
    }))


output_data = {
    "meta": {
        "dataset": DATA_PATH,
        "target_column": target_col,
        "future_years": FUTURE_YEARS,
        "test_size": TEST_SIZE,
        "n_splits": N_SPLITS,
        "last_n_years_used_per_country": LAST_N_YEARS,
        "use_log_target_for_ml_models": USE_LOG_TARGET,
        "candidate_models": candidate_models,
        "sarimax_order": SARIMAX_ORDER,
        "sarimax_seasonal_order": SARIMAX_SEASONAL,
        "drop_accuracy_threshold": ACC_THRESHOLD,
        "accuracy_method": "wMAPE-based CV median",
        "augmentation_used": False
    },
    "dropped_countries": dropped_countries,
    "predictions": predictions_list,
    "drop_reasons_summary": drop_reasons
}

with open(JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 70)
print(" Dropped Countries:\n")
for c in dropped_countries:
    if c.get("accuracy") is None:
        print(f"{c['country']} --> None ({c['reason']})")
    else:
        bm = c.get("best_model_tried", "N/A")
        print(f"{c['country']} --> {c['accuracy']:.2f}% ({c['reason']}, best={bm})")

print(f"\nTotal Dropped: {len(dropped_countries)}")
print("\nDrop Reasons Summary:")
for k, v in drop_reasons.items():
    print(f"{k}: {v}")

print(f"\nJSON file saved to: {JSON_PATH}")

Target: Annual nitrous oxide emissions in CO₂ equivalents
Shape after cleaning: (35175, 4)
Countries: 201
XGBoost available: True
Country: Afghanistan
Selected model: sarimax
Years used: 50
CV folds used: 3
Accuracy (CV median): 95.70%
   Year  Predicted_nitrous_emissions
0  2025                 4.316442e+06
1  2026                 4.318861e+06
2  2027                 4.318664e+06
3  2028                 4.318680e+06
4  2029                 4.318679e+06
Country: Albania
Selected model: ridge
Years used: 50
CV folds used: 3
Accuracy (CV median): 95.19%
   Year  Predicted_nitrous_emissions
0  2025                846595.637517
1  2026                905724.981167
2  2027                904210.202801
3  2028                909972.590349
4  2029                920306.631096


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Country: Algeria
Selected model: sarimax
Years used: 50
CV folds used: 3
Accuracy (CV median): 94.12%
   Year  Predicted_nitrous_emissions
0  2025                 8.125693e+06
1  2026                 8.127617e+06
2  2027                 8.127947e+06
3  2028                 8.128003e+06
4  2029                 8.128013e+06
Country: Andorra
Selected model: xgb
Years used: 50
CV folds used: 3
Accuracy (CV median): 95.49%
   Year  Predicted_nitrous_emissions
0  2025                 38078.522362
1  2026                 37795.591767
2  2027                 37649.924398
3  2028                 37439.526577
4  2029                 37439.526577
Country: Angola
Selected model: rf
Years used: 50
CV folds used: 3
Accuracy (CV median): 96.34%
   Year  Predicted_nitrous_emissions
0  2025                 1.735977e+07
1  2026                 1.741278e+07
2  2027                 1.749056e+07
3  2028                 1.746968e+07
4  2029                 1.732164e+07


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Country: Antigua and Barbuda
Selected model: xgb
Years used: 50
CV folds used: 3
Accuracy (CV median): 96.18%
   Year  Predicted_nitrous_emissions
0  2025                 13049.000180
1  2026                 13547.878655
2  2027                 13623.523105
3  2028                 13533.479154
4  2029                 13178.561518
Country: Argentina
Selected model: ridge
Years used: 50
CV folds used: 3
Accuracy (CV median): 96.08%
   Year  Predicted_nitrous_emissions
0  2025                 4.977309e+07
1  2026                 5.028678e+07
2  2027                 5.079911e+07
3  2028                 5.132551e+07
4  2029                 5.192073e+07
Country: Armenia
Selected model: xgb
Years used: 50
CV folds used: 3
Accuracy (CV median): 92.83%
   Year  Predicted_nitrous_emissions
0  2025                653922.180783
1  2026                648719.465308
2  2027                700501.185657
3  2028                747799.043472
4  2029                799137.475115
Country: Aruba
Selected 

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


... (29 outputs left out)

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Country: Sweden
Selected model: rf
Years used: 50
CV folds used: 3
Accuracy (CV median): 97.55%
   Year  Predicted_nitrous_emissions
0  2025                 6.117224e+06
1  2026                 6.154734e+06
2  2027                 6.168542e+06
3  2028                 6.190425e+06
4  2029                 6.191434e+06
Country: Switzerland
Selected model: rf
Years used: 50
CV folds used: 3
Accuracy (CV median): 97.67%
   Year  Predicted_nitrous_emissions
0  2025                 2.188922e+06
1  2026                 2.191248e+06
2  2027                 2.192893e+06
3  2028                 2.191557e+06
4  2029                 2.192822e+06
Country: Syria
Selected model: xgb
Years used: 50
CV folds used: 3
Accuracy (CV median): 94.75%
   Year  Predicted_nitrous_emissions
0  2025                 4.071225e+06
1  2026                 4.303291e+06
2  2027                 4.194959e+06
3  2028                 4.171529e+06
4  2029                 4.374519e+06
Country: Taiwan
Selected model: rf
Years 

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Country: United Kingdom
Selected model: ridge
Years used: 50
CV folds used: 3
Accuracy (CV median): 96.96%
   Year  Predicted_nitrous_emissions
0  2025                 2.722329e+07
1  2026                 2.689853e+07
2  2027                 2.658504e+07
3  2028                 2.628960e+07
4  2029                 2.599577e+07
Country: United States
Selected model: sarimax
Years used: 50
CV folds used: 3
Accuracy (CV median): 98.36%
   Year  Predicted_nitrous_emissions
0  2025                 2.395978e+08
1  2026                 2.395979e+08
2  2027                 2.395979e+08
3  2028                 2.395979e+08
4  2029                 2.395979e+08
Country: Uruguay
Selected model: xgb
Years used: 50
CV folds used: 3
Accuracy (CV median): 98.54%
   Year  Predicted_nitrous_emissions
0  2025                 7.517588e+06
1  2026                 7.324643e+06
2  2027                 7.224335e+06
3  2028                 7.228339e+06
4  2029                 7.319224e+06
Country: Uzbekistan
S